# Train BASELINE MF — ML-10M (lưu weight lên Drive)

Baseline **MF** (Matrix Factorization) — model học thật, cá nhân hóa, dùng để so sánh với DIVRS (giống paper).
MF dùng negative sampling đều → **nhanh hơn DIVRS** nhiều. Weight ghi thẳng vào **Drive/Colab Notebooks** mỗi epoch.
⚙️ Bật **T4 GPU**.

## 1. GPU + lib

In [ ]:
!nvidia-smi || echo 'KHONG co GPU -> chay CPU (cham)'
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Code + data + Drive

In [ ]:
import os
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
os.makedirs(OUT_DIR, exist_ok=True)
print('Luu vao:', OUT_DIR)

## 3. Train MF + Test
`epochs=30`, `es_patience=5`. Ghi checkpoint mỗi epoch thẳng lên Drive (ngắt phiên vẫn còn).

In [ ]:
USE_GPU=torch.cuda.is_available(); print('use_gpu =', USE_GPU)
!python app.py \
  --flagfile config/ml10m_mf.cfg \
  --output "{OUT_DIR}/" \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=30 --es_patience=5

## 4. Xem kết quả + checkpoint (đọc thẳng từ Drive)

In [ ]:
import glob
# run MF moi nhat (folder khong chua 'divrs')
run=max([r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' not in r.lower()], key=os.path.getmtime)
print('Run MF:', run)
print('===== TEST metrics =====')
!find "{run}test_log" -type f -exec grep -hE "TEST results|recall|hit_ratio|ndcg" {{}} + 2>/dev/null
print('===== Checkpoints tren Drive =====')
!ls -lah "{run}ckpt/" 2>/dev/null

## Ghi chú
- Xong cái này, mở `demo_DIVRS.ipynb` chạy lại — nó **tự phát hiện run MF** và dùng làm baseline (thay Most-Popular).
- MF là baseline **có học + cá nhân hóa** nhưng vẫn dính popularity bias → tương phản với DIVRS sát thực tế hơn Most-Popular.